<a href="https://colab.research.google.com/github/Sunil032003/deep-learning-projects/blob/main/Ag_news.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This is the lab Experiment using simple RNN

In [20]:
!pip install datasets

In [21]:
from datasets import load_dataset
raw_datasets = load_dataset("wangrongsheng/ag_news")

In [22]:
import pandas as pd
import numpy as np


In [23]:
df=pd.DataFrame(raw_datasets['train'])

In [24]:
df.head()

,text,label
0,Wall St. Bears Claw Back Into the Black (Reute...,2
1,Carlyle Looks Toward Commercial Aerospace (Reu...,2
2,Oil and Economy Cloud Stocks' Outlook (Reuters...,2
3,Iraq Halts Oil Exports from Main Southern Pipe...,2
4,"Oil prices soar to all-time record, posing new...",2


In [25]:
## Regular Expression use to find pattern and replace or modify text
import re
def clean(text):
    text = text.lower() ## converts text into lowercase
     # Remove numbers, punctuation, and special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text

In [26]:
df["cleaned_news"]=df["text"].apply(clean) ## create new column to store cleaned text

Tokenizers is used to convert tect into *numbers*

In [27]:
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer(num_words=10000) ## words only that appears frequently

# Build the vocabulary from the cleaned text

tokenizer.fit_on_texts(df["cleaned_news"].astype(str))

# Convert each sentence into a sequence of integers

sequences = tokenizer.texts_to_sequences(df["cleaned_news"].astype(str))

In [28]:
print(type(sequences))

<class 'list'>


In [29]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Make all sequences the same length (50 words)
X = pad_sequences(
    sequences,
    maxlen=50,
    padding="post",
    truncating="post"
)

In [30]:
print(X.shape)

(120000, 50)


In [31]:
from tensorflow.keras.utils import to_categorical
# Convert class labels into one-hot encoded vectors
y = to_categorical(df["label"])

In [32]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding
from tensorflow.keras.layers import SimpleRNN
from tensorflow.keras.layers import Dense

In [34]:
model = Sequential()

model.add(Embedding(
    input_dim=10000,  # Vocabulary size
    output_dim=128,   # Embedding vector size
    input_shape=(50,)   # Sequence length is 50
))

model.add(SimpleRNN(64))

model.add(Dense(4, activation="softmax"))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:103: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [35]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 50, 128)        │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ (None, 64)             │        12,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,292,612 (4.93 MB)

 Trainable params: 1,292,612 (4.93 MB)

 Non-trainable params: 0 (0.00 B)

In [36]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [39]:
history = model.fit(
    X,
    y,
    epochs=5, # Number of training iterations
    batch_size=32, # Number of samples per batch
    validation_split=0.2 # 20% data used for validation
)

Epoch 1/5
3000/3000 ━━━━━━━━━━━━━━━━━━━━ 20s 7ms/step - accuracy: 0.9566 - loss: 0.1413 - val_accuracy: 0.8322 - val_loss: 0.6537
Epoch 2/5
3000/3000 ━━━━━━━━━━━━━━━━━━━━ 19s 6ms/step - accuracy: 0.9696 - loss: 0.0982 - val_accuracy: 0.8400 - val_loss: 0.5963
Epoch 3/5
3000/3000 ━━━━━━━━━━━━━━━━━━━━ 19s 6ms/step - accuracy: 0.9656 - loss: 0.1136 - val_accuracy: 0.8508 - val_loss: 0.5648
Epoch 4/5
3000/3000 ━━━━━━━━━━━━━━━━━━━━ 21s 7ms/step - accuracy: 0.9637 - loss: 0.1170 - val_accuracy: 0.8425 - val_loss: 0.5872
Epoch 5/5
3000/3000 ━━━━━━━━━━━━━━━━━━━━ 19s 6ms/step - accuracy: 0.9623 - loss: 0.1194 - val_accuracy: 0.8346 - val_loss: 0.6451


In [40]:
loss, accuracy = model.evaluate(X, y)

print("Loss :", loss)
print("Accuracy :", accuracy)

3750/3750 ━━━━━━━━━━━━━━━━━━━━ 12s 3ms/step - accuracy: 0.9334 - loss: 0.2434
Loss : 0.2434089034795761
Accuracy : 0.9334083199501038
